# D07 · CAN filters, error confinement and termination

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D07_can_filters_error_confinement_and_termination.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 50 min

**Skill Hub concepts:** CAN acceptance filters and masks; CAN error confinement; CAN error frame and remote frame; CAN termination

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D07")

## Map: what the controller does with every frame and every error

**Acceptance filter:** accept when `(id & mask) == (filter & mask)`; mask bit 1 = "must match", 0 = "don't care".

**Error confinement** (per node, ISO 11898-1, simplified):

| Event | Counter change |
|---|---|
| transmit error | TEC += 8 |
| successful transmit | TEC -= 1 (not below 0) |
| receive error | REC += 1 |
| successful receive | REC -= 1 (not below 0); if REC > 127 it drops back to 127 |

```
  ERROR ACTIVE ──(TEC>127 or REC>127)──> ERROR PASSIVE ──(TEC>255)──> BUS OFF
  sends 6 dominant bits             sends 6 recessive bits        silent; 128 x 11 recessive bits
  as error flag                     (cannot disturb others)       then back to ERROR ACTIVE, counters 0
     ^────────────(both <= 127)──────────┘
```

| Frame | What it is | Note |
|---|---|---|
| Error frame | error flag (6 bits) + 8-bit delimiter, sent the moment a node sees an error | violates stuffing on purpose so everyone notices |
| Remote frame | RTR = 1, no data, DLC = requested length: "whoever owns this ID, send it" | removed in CAN FD; most designs use periodic or request/response IDs instead |

**Termination:** 120 ohm at each physical end, so the line looks like its characteristic impedance (about 120 ohm) and
edges do not reflect. Powered off, CANH to CANL measures the parallel combination: two resistors = 60 ohm.

## Task 1: filters

In [ ]:
def filter_mask_for(ids):
    """Tightest single (filter, mask) pair (11-bit) that accepts every id in `ids`."""
    raise NotImplementedError  # TODO

def accepted(filt, mask):
    raise NotImplementedError  # TODO

lab.check("filters", filter_mask_for, accepted)

In [ ]:
%%writefile d07_filters.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d07.h
// Grade it with lab.check("filters", "d07_filters.c"). For C++, name the file .cpp in both places.
#include "d07.h"

void can_filter_mask_for(const uint16_t *ids, size_t n, uint16_t *filt, uint16_t *mask) {
    (void)ids; (void)n;
    *filt = 0; *mask = 0; /* TODO */
}

size_t can_accepted(uint16_t filt, uint16_t mask, uint16_t *out, size_t out_cap) {
    (void)filt; (void)mask; (void)out; (void)out_cap;
    return 0; /* TODO */
}

In [ ]:
# lab.check("filters", "d07_filters.c")  # uncomment after writing the C/C++ cell above

## Task 2: the error-confinement state machine

In [ ]:
class ErrorCounters:
    def __init__(self):
        self.tec = 0
        self.rec = 0
        self._recov = 0

    @property
    def state(self):
        raise NotImplementedError  # TODO

    def tx_error(self):
        if self.state != "bus-off":
                        pass  # TODO

    def tx_ok(self):
        pass  # TODO

    def rx_error(self):
        if self.state != "bus-off":
                        pass  # TODO

    def rx_ok(self):
        pass  # TODO

    def recessive_11(self):
        """Called for each run of 11 recessive bits seen while bus-off."""
        pass  # TODO

lab.check("confinement", ErrorCounters)

In [ ]:
%%writefile d07_confinement.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d07.h
// Grade it with lab.check("confinement", "d07_confinement.c"). For C++, name the file .cpp in both places.
#include "d07.h"

void cec_init(CanErrorCounters *c) { c->tec = c->rec = c->recov = 0; }
int cec_state(const CanErrorCounters *c) { (void)c; return CAN_ERROR_ACTIVE; /* TODO */ }
void cec_tx_error(CanErrorCounters *c) { (void)c; /* TODO */ }
void cec_tx_ok(CanErrorCounters *c) { (void)c; /* TODO */ }
void cec_rx_error(CanErrorCounters *c) { (void)c; /* TODO */ }
void cec_rx_ok(CanErrorCounters *c) { (void)c; /* TODO */ }
void cec_recessive_11(CanErrorCounters *c) { (void)c; /* TODO */ }

In [ ]:
# lab.check("confinement", "d07_confinement.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** Filter 0x120, mask 0x7F0. How many standard identifiers pass?

**M2 (Apply).** From TEC = 0 and only transmit errors, how many errors until bus-off?

**M3 (Analyze).** Someone added a third 120 ohm terminator in the middle of the bus. Ohms measured between CANH and CANL
with power off?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A node with a broken transceiver corrupts every frame it sends. What protects the rest of the bus?
- a) nothing; the bus is down until someone unplugs it
- b) its TEC climbs by 8 per failure, it goes error-passive (cannot destroy others' frames) and then bus-off
- c) the other nodes vote it out with remote frames

**K2 (Evaluate).** Your Storm board's CAN works on the bench with a 30 cm cable but throws errors on the robot.
Give a troubleshooting order covering termination, bit timing and the error counters.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d07.h`).
3. In `roboprep/checks/d07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()